# TamilEOT — fine-tuning Smart Turn v3 for Tamil

**Runtime → Change runtime type → T4 GPU** before running anything.

Baseline to beat, measured on the sealed 30-call test set:

| | accuracy | FPR (std) | ROC-AUC |
|---|---|---|---|
| always say `complete` | 63.08% | 100% | 0.500 |
| `smart-turn-v3.0` zero-shot | 65.64% | 19.17% | 0.779 |
| `smart-turn-v3.2` zero-shot | **70.30%** | 36.39% | 0.751 |

For scale: Smart Turn's published figures are 94.26% English, 82.43% Marathi
(its worst of 23 languages). Tamil is not among them.

**FP = the agent talks over the user.** That is the error that matters; do not
read accuracy alone.

### Dataset — 18,485 clips from 116 real Tamil calls

| split | clips | complete | incomplete | note |
|---|---|---|---|---|
| train | 11,992 | 7,908 | 4,084 | +1,987 from the relaxed funnel |
| dev | 2,325 | 1,532 | 793 | +285 |
| test | 4,168 | 2,629 | 1,539 | **frozen — byte-identical to every prior run** |

The extra 2,272 rows come from `pipeline/07_relax_funnel.py`, a second pass
over the 34,316 boundaries the first-pass gates rejected. The test split was
deliberately left out of that harvest so this run stays comparable with the
83.54% the previous checkpoint scored. See `reports/funnel_relaxed.md`.

### Measured so far, all on this frozen test set

| encoder | lr | accuracy | AUC | FP/N | FN/N |
|---|---|---|---|---|---|
| `smart-turn-v3.2` zero-shot | – | 70.30% | 0.751 | – | – |
| whisper-tiny | 1.4e-5 | 83.57% | 0.901 | 10.10% | 6.33% |
| whisper-tiny | 5e-5 | 83.35% | 0.904 | 7.73% | 8.90% |
| **whisper-base** | **5e-5** | **86.23%** | **0.922** | **8.95%** | **4.80%** |

**Capacity was the constraint, not data.** More data (+20% rows), the funnel
harvest, and the learning rate each moved AUC by ~0.003. Swapping tiny for base
moved it **+0.018** and accuracy **+2.88**.

Note what ships, though: Smart Turn v3 *is* whisper-tiny, so only a tiny
fine-tune is a drop-in replacement. base is the diagnostic that priced the
tradeoff at 2.9 points — hand Sarvam both and let them choose.

`seed_everything()` in §3 pins cuDNN as well as the RNGs. Without it, four
unseeded runs spanned 2.2 points and no single-run comparison meant anything.

---

### Memory

Colab gives ~12 GB. Held naively this dataset does not fit: the raw audio is
4.15 GB, the features another 4.16 GB, and an ONNX batch of 256 allocates a
~1 GB attention tensor *per layer*. So nothing here is held whole —

* the FLAC is **streamed** block by block, never decoded into RAM at once,
* features are written to a **disk memmap** as float32, so they cost page cache (which the
  kernel reclaims under pressure) rather than heap (which OOMs),
* inference runs at batch 64 and converts fp16 → fp32 one batch at a time.

Peak stays in the hundreds of MB. Every cell prints RSS so you can see it.

### Two traps, both already hit

1. **The Whisper encoder is built for 30 s, this is 8 s.** Its positional
   embedding table has 1500 rows; 8 s of mel gives 400 positions, and the stock
   encoder adds all 1500 and crashes.
2. **Nothing here returns a logit.** Upstream's `forward` returns
   `sigmoid(...)` under a key called `"logits"`, and the ONNX graph ends in a
   `Sigmoid` for the same reason. Applying another sigmoid maps [0,1] onto
   [0.5, 0.73] and makes every clip predict `complete` — a wrong answer that
   looks entirely plausible.

In [ ]:
!nvidia-smi -L
!pip -q install 'transformers>=4.44' soundfile scikit-learn onnxruntime
import os, gc, json, time, random, numpy as np, torch, transformers

def ram(tag=''):
    kb = int(open('/proc/self/status').read().split('VmRSS:')[1].split()[0])
    free = int([l for l in open('/proc/meminfo') if l.startswith('MemAvailable')][0].split()[1])
    print(f'  [RAM] {tag:28s} process {kb/1e6:5.2f} GB   available {free/1e6:5.2f} GB')

print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| transformers', transformers.__version__)
ram('start')

## 1. Data

Copy the FLAC bundles off Drive onto Colab's **local** disk first. Reading them
in place through the Drive FUSE mount is many times slower, and this is three
big sequential files, which is the one thing Drive is good at.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS if you put the folder somewhere else
SRC = '/content/drive/MyDrive/TamilEOT/colab'

!mkdir -p /content/data /content/feats
!cp {SRC}/tamileot_*.flac {SRC}/tamileot_index.jsonl /content/data/
!ls -la /content/data
!df -h /content | tail -1

In [ ]:
import soundfile as sf

SR, CLIP = 16_000, 8 * 16_000

idx = [json.loads(l) for l in open('/content/data/tamileot_index.jsonl')]
by_split = {}
for r in idx:
    by_split.setdefault(r['split'], []).append(r)
for v in by_split.values():
    v.sort(key=lambda r: r['i'])

for split in ('train', 'dev', 'test'):
    rows = by_split[split]
    info = sf.info(f'/content/data/tamileot_{split}.flac')
    # clip i sits at exactly [i*CLIP : (i+1)*CLIP] -- verify, do not trust
    assert info.samplerate == SR and info.frames == len(rows) * CLIP, 'stream/index mismatch'
    assert [r['i'] for r in rows] == list(range(len(rows))), 'index is not contiguous'
    n1 = sum(r['endpoint_bool'] for r in rows)
    nr = sum(r.get('harvest', 'core') == 'relaxed' for r in rows)
    print(f'{split:6s} {len(rows):6,d} clips   {n1:,} complete / {len(rows)-n1:,} incomplete'
          f'   ({nr:,} from the relaxed funnel)')

# The test split is the benchmark every prior number was measured on. The 03c
# harvest deliberately left its 802 rows unlabelled so it could not move; if a
# future repack ever lets them in, every comparison silently stops being one.
assert sum(r.get('harvest', 'core') == 'relaxed' for r in by_split['test']) == 0, \
    'test split contains relaxed rows -- the benchmark moved, old numbers no longer compare'
assert len(by_split['test']) == 4168, f"test is {len(by_split['test'])}, expected 4,168"

calls = {s: {r['call'] for r in v} for s, v in by_split.items()}
assert not (calls['train'] & calls['test']) and not (calls['train'] & calls['dev'])
print('\nno call appears in two splits — split integrity holds')
ram('index loaded')

## 2. Features

`do_normalize=True` is **not** the Whisper default, and Smart Turn's
`inference.py` passes it. Getting this wrong changes what the model hears and
quietly invalidates every comparison against the baseline above.

The FLAC is read as a stream and the output goes straight to a memmap on disk,
so neither the 4.15 GB of audio nor the 4.16 GB of features is ever resident.
Takes a few minutes; done once.

In [ ]:
from transformers import WhisperFeatureExtractor

FE = WhisperFeatureExtractor(chunk_length=8)   # 8 s -> (80, 800)

def feats(batch_int16, n_samples):
    """int16 [B, 128000] -> float32 [B, 80, 800], exactly as Smart Turn does it."""
    waves = [b[:n].astype(np.float32) / 32768.0 for b, n in zip(batch_int16, n_samples)]
    out = FE(waves, sampling_rate=SR, return_tensors='np',
             padding='max_length', max_length=CLIP, truncation=True, do_normalize=True)
    return out['input_features'].astype(np.float32)

B = 64
FEATS, PATHS = {}, {}
for split in ('train', 'dev', 'test'):
    rows = by_split[split]
    path = f'/content/feats/{split}.f16'
    PATHS[split] = (path, (len(rows), 80, 800))
    if not os.path.exists(path):
        mm = np.memmap(path, dtype=np.float32, mode='w+', shape=(len(rows), 80, 800))
        t0 = time.time()
        with sf.SoundFile(f'/content/data/tamileot_{split}.flac') as f:
            for i in range(0, len(rows), B):
                n = min(B, len(rows) - i)
                blk = f.read(n * CLIP, dtype='int16').reshape(n, CLIP)
                mm[i:i+n] = feats(blk, [r['n_samples'] for r in rows[i:i+n]])
                if i % (B * 20) == 0:
                    print(f'\r  {split} {i+n:,}/{len(rows):,}', end='')
        mm.flush(); del mm; gc.collect()
        print(f'\r  {split}: {os.path.getsize(path)/1e9:.2f} GB in {time.time()-t0:.0f}s')
    FEATS[split] = np.memmap(path, dtype=np.float32, mode='r', shape=(len(rows), 80, 800))

assert FEATS['test'].shape[1:] == (80, 800)
ram('features on disk')

### Preprocessing canary — do not delete this

Not a result: a check that the features built above are the ones Smart Turn
expects. `transformers` is unpinned here, and if `WhisperFeatureExtractor` ever
changes how `do_normalize` or padding behaves, every number in this notebook
shifts and the exported model would consume features Pipecat never produces.

Runs on **every 8th test clip after sorting by `sid`** — 521 of 4,168, about ten
seconds. Sorted by `sid`, not by position: the index and `samples_llm.jsonl`
hold the same clips in different orders, so a positional stride picks a
different sample on each side and the comparison is meaningless (measured: 60 of
521 clips in common).

**Asserted on AUC, not accuracy.** The two pipelines disagree on ~1.5% of clips
that sit near p=0.5 and flip on float noise — harmless, it cancels out over the
full split (70.30% here vs 70.27% measured in Colab), but it moves subset
accuracy by a point or more. AUC is threshold-free and ignores it. The failures
worth catching are not subtle: a broken extractor gives AUC ~0.5, never 0.75.


In [ ]:
!wget -q -nc https://huggingface.co/pipecat-ai/smart-turn-v3/resolve/main/smart-turn-v3.2-cpu.onnx
import onnxruntime as ort
from sklearn.metrics import roc_auc_score

STRIDE = 8
pick = sorted(by_split['test'], key=lambda r: r['sid'])[::STRIDE]
Xc = np.ascontiguousarray(FEATS['test'][[r['i'] for r in pick]], dtype=np.float32)
yc = np.array([r['endpoint_bool'] for r in pick])

so = ort.SessionOptions()
so.enable_cpu_mem_arena = False      # do not let the arena keep growing
sess = ort.InferenceSession('smart-turn-v3.2-cpu.onnx', so,
                            providers=['CPUExecutionProvider'])
nm = sess.get_inputs()[0].name
p = np.concatenate([sess.run(None, {nm: Xc[i:i + 64]})[0].reshape(-1)
                    for i in range(0, len(Xc), 64)])
# the graph's output is *named* logits and is not one -- the final node is a
# Sigmoid. Applying another maps [0,1] onto [0.5, 0.73] and makes every clip
# look complete, which scores exactly the majority baseline and looks plausible.
assert 0.0 <= p.min() and p.max() <= 1.0, 'output is not a probability'

acc, a = 100 * ((p > 0.5).astype(int) == yc).mean(), roc_auc_score(yc, p)
print(f'zero-shot v3.2 on sid-sorted test[::{STRIDE}]   n={len(Xc)}')
print(f'  AUC      {a:.4f}    (local pipeline: 0.8070 — assert is > 0.75)')
print(f'  accuracy {acc:5.2f}%   (local pipeline: 72.74% — +-1.5 is float noise, not a fault)')
print(f'  mean p   {p.mean():.4f}    (local pipeline: 0.5682)')
assert a > 0.75, f'AUC {a:.4f} — these are not the features Smart Turn was trained on'
print('preprocessing matches the ruler the baseline was measured with')

del sess, Xc, p; gc.collect()
ram('canary')


## 3. Model

This mirrors `SmartTurnV3Model` in `pipecat-ai/smart-turn`'s `train.py`
verbatim — same encoder, same attention pooling, same classifier stack, same
per-batch `pos_weight`. Matching it exactly is the point: the resulting state
dict and ONNX graph drop straight into their tooling, and any Tamil number is
then comparable to their published ones.

`config.max_source_positions = 400` is the fix for the 1500-vs-400 crash.

In [ ]:
import torch.nn as nn
from torch.nn.functional import softmax
from transformers import WhisperPreTrainedModel, WhisperConfig, WhisperModel
from transformers.models.whisper.modeling_whisper import WhisperEncoder

class SmartTurnV3Model(WhisperPreTrainedModel):
    def __init__(self, config: WhisperConfig):
        super().__init__(config)
        config.max_source_positions = 400        # 8 s, not 30 s
        self.encoder = WhisperEncoder(config)
        h = config.d_model
        self.pool_attention = nn.Sequential(nn.Linear(h, 256), nn.Tanh(), nn.Linear(256, 1))
        self.classifier = nn.Sequential(
            nn.Linear(h, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, 64), nn.GELU(), nn.Linear(64, 1))
        # transformers 5 requires this; 4.x tolerated its absence. Called before
        # the manual init below so upstream's std=0.1 heads still win.
        self.post_init()
        for seq in (self.classifier, self.pool_attention):
            for m in seq:
                if isinstance(m, nn.Linear):
                    m.weight.data.normal_(mean=0.0, std=0.1)
                    if m.bias is not None:
                        m.bias.data.zero_()

    def forward(self, input_features, labels=None):
        z = self.encoder(input_features=input_features).last_hidden_state
        w = softmax(self.pool_attention(z), dim=1)
        logits = self.classifier(torch.sum(z * w, dim=1))
        if labels is not None:
            pw = ((labels == 0).sum() / (labels == 1).sum().clamp(min=1)).clamp(0.1, 10.0)
            loss = nn.BCEWithLogitsLoss(pos_weight=pw)(logits.view(-1), labels.float().view(-1))
            return {'loss': loss, 'logits': torch.sigmoid(logits.detach())}
        return {'logits': torch.sigmoid(logits)}   # a probability, despite the name

dev = 'cuda' if torch.cuda.is_available() else 'cpu'

# whisper-tiny is the shipping architecture -- Smart Turn v3 *is* tiny, so a
# tiny fine-tune is a drop-in replacement and a base one is not. base is here
# because it answers a question tiny cannot: on 2026-08-22 it scored 86.23%
# against tiny's 83.35% on the same frozen test set, +0.018 AUC. Every other
# lever tried (more data, funnel harvest, learning rate) moved AUC by ~0.003.
# Capacity was the binding constraint, and that is worth knowing even if the
# artefact you ship stays tiny.
ENCODER = 'openai/whisper-base'      # 'openai/whisper-tiny' for the shippable one

def fresh_model():
    """A new model with pretrained encoder weights and random heads.

    The learning curve needs an untouched model per fraction -- reusing one
    that has already seen the full training set would measure nothing.

    Size-agnostic on purpose: both heads read `config.d_model`, and
    `max_source_positions` is pinned to 400 in `__init__`, so tiny (384-dim,
    4 layers) and base (512-dim, 6 layers) both work with no other change.
    Mel is 80 bins for both, so the feature cache does NOT need rebuilding
    when you switch -- skip §2 and save the extraction."""
    m = SmartTurnV3Model.from_pretrained(
        ENCODER, num_labels=1, ignore_mismatched_sizes=True)
    assert m.encoder.embed_positions.num_embeddings == 400
    full = WhisperModel.from_pretrained(ENCODER)
    with torch.no_grad():
        m.encoder.embed_positions.weight.copy_(full.encoder.embed_positions.weight[:400])
    assert torch.equal(m.encoder.embed_positions.weight.data,
                       full.encoder.embed_positions.weight.data[:400])
    assert torch.equal(m.encoder.conv1.weight.data, full.encoder.conv1.weight.data), \
        'pretrained encoder weights did not load'
    del full; gc.collect()
    return m.to(dev)

SEED = 0

def seed_everything(seed=SEED):
    """Every source of run-to-run movement, not just torch's CPU generator.

    `torch.manual_seed` alone still left the cuDNN convolution algorithm free
    to be chosen per run, and the Whisper encoder opens with two convs. Four
    unseeded runs of the training cell spanned 82.10-84.29% on test -- a 2.2
    point range, wider than any effect measured since, which made every
    single-run comparison meaningless.

    `cudnn.deterministic` costs some speed. That is the right trade here: the
    funnel relaxation is worth +0.5 to +1.2 points and cannot be detected
    against a 2.2 point spread.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()           # seeds the random classifier/pooling heads
model = fresh_model()

with torch.no_grad():
    _o = model(torch.from_numpy(np.ascontiguousarray(FEATS['dev'][:2], np.float32)).to(dev))['logits']
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M params on {dev}; "
      f"smoke test {tuple(_o.shape)} in [{_o.min():.3f}, {_o.max():.3f}]")
ram('model loaded')

## 4. Train

Upstream uses batch 384 / 4 epochs / lr 5e-5 / cosine / warmup 0.2 on ~270k
samples. This is 10k samples, so batch 32 — and **the learning rate has to come
down with it**. Keeping 5e-5 at batch 32 is 12x fewer samples per step at the
same step size; it bounces, peaks at epoch 2 and decays. Square-root scaling
gives `1.4e-5`, which measured +2.24 points on dev `hold_intra` and converges
smoothly. Linear scaling to `4e-6` overcorrects and is still undertrained at 6
epochs. §5c is the sweep.

**Select on `dev`, and touch `test` once at the end.**


In [ ]:
from torch.utils.data import Dataset, DataLoader

LABEL = 'endpoint_bool'      # or 'label_pipeline' / 'llm_verdict' to compare policies
UNDISPUTED_ONLY = False      # True -> train only where both witnesses agree

class Clips(Dataset):
    """Reads from the on-disk memmap. Workers inherit the mapping on fork, so
    the features cost page cache rather than one heap copy per worker."""
    def __init__(self, split, undisputed=False):
        self.rows = [r for r in by_split[split] if not (undisputed and r['dispute'])]
        self.path, self.shape = PATHS[split]
        self.X = None
        self.ix = np.array([r['i'] for r in self.rows])
        self.y = np.array([float(r[LABEL]) for r in self.rows], dtype=np.float32)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, k):
        if self.X is None:      # open lazily, once per worker process
            self.X = np.memmap(self.path, dtype=np.float32, mode='r', shape=self.shape)
        x = np.ascontiguousarray(self.X[self.ix[k]], dtype=np.float32)
        return torch.from_numpy(x), torch.tensor(self.y[k])

# `shuffle=True` draws from torch's global generator and each worker reseeds
# numpy from a base seed drawn the same way -- both are seeded above, but
# passing them explicitly means the run does not depend on how many other cells
# happened to consume random numbers first.
_g = torch.Generator(); _g.manual_seed(SEED)

def _worker_seed(k):
    np.random.seed(SEED + k); random.seed(SEED + k)

tr = DataLoader(Clips('train', UNDISPUTED_ONLY), batch_size=32, shuffle=True,
                num_workers=2, pin_memory=True, drop_last=True,
                generator=_g, worker_init_fn=_worker_seed)
dv = DataLoader(Clips('dev'), batch_size=64, num_workers=2)
nrel = sum(r.get('harvest', 'core') == 'relaxed' for r in tr.dataset.rows)
print(f'{len(tr.dataset):,} train / {len(dv.dataset):,} dev   label={LABEL}'
      f'   ({nrel:,} train rows from the relaxed funnel)')
ram('dataloaders')

In [ ]:
import shutil
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

EPOCHS, LR = 6, 5e-5       # §5c re-picked 5e-5 once train grew to 11,992.
                           # The sqrt-scaled 1.4e-5 was compensating for a set
                           # small enough that 5e-5 overfitted; it no longer is.
TAG = f"{ENCODER.rsplit('-', 1)[-1]}_{LR:.0e}"      # e.g. base_5e-05
seed_everything()          # shuffle + dropout; the head init is seeded in §3.
                           # Unseeded, four runs of this cell spanned 82.10-84.29%
                           # on test, and no single-run comparison meant anything.
opt = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = len(tr) * EPOCHS
sched = get_cosine_schedule_with_warmup(opt, int(0.2 * steps), steps)
scaler = torch.amp.GradScaler('cuda', enabled=(dev == 'cuda'))

@torch.no_grad()
def evaluate(loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        with torch.amp.autocast('cuda', enabled=(dev == 'cuda')):
            P.append(model(x.to(dev, non_blocking=True))['logits'].float().cpu())
        Y.append(y)
    p, y = torch.cat(P).view(-1).numpy(), torch.cat(Y).numpy()
    pred = (p > 0.5).astype(int)
    fp = int(((pred == 1) & (y == 0)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    return (pred == y).mean(), fp / max(fp + tn, 1), p, y

best = 0.0
for ep in range(EPOCHS):
    model.train()
    for k, (x, y) in enumerate(tr):
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(dev == 'cuda')):
            loss = model(x.to(dev, non_blocking=True), labels=y.to(dev))['loss']
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        if k % 50 == 0:
            print(f'\r  ep{ep} {k}/{len(tr)} loss {loss.item():.4f}', end='')
    acc, fpr, _, _ = evaluate(dv)
    print(f'\r  epoch {ep}: dev acc {100*acc:.2f}%  FPR {100*fpr:.2f}%   ', end='')
    if acc > best:
        best = acc
        torch.save(model.state_dict(), '/content/best.pt')
        # ...and straight to Drive. The 2026-08-22 session hit its usage limit
        # moments after training and took /content with it; only what was
        # already on Drive survived, and an LR sweep winner was lost that way.
        # 32 MB per improvement is cheap insurance against losing the run.
        shutil.copy('/content/best.pt', f'{SRC}/best_{TAG}.pt')
        print(f'saved -> {SRC}/best_{TAG}.pt')
    else:
        print()
print(f'\nbest dev accuracy {100*best:.2f}%')
ram('after training')

## 5. Test — run once

30 calls that have never been trained on. Report accuracy **and** FPR, and
compare against 70.30% / 36.39% zero-shot.

In [ ]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

model.load_state_dict(torch.load('/content/best.pt'))
te = DataLoader(Clips('test'), batch_size=64, num_workers=2)
acc, fpr, p, y = evaluate(te)
pr, rc, f1, _ = precision_recall_fscore_support(y, (p > 0.5).astype(int), average='binary')

print(f'TEST  accuracy {100*acc:.2f}%   FPR {100*fpr:.2f}%   AUC {roc_auc_score(y, p):.3f}')
print(f'      precision {pr:.3f}  recall {rc:.3f}  F1 {f1:.3f}')
print(f'\nzero-shot v3.2:  70.30%  /  FPR 36.39%  /  AUC 0.751')
print(f'delta:          {100*acc-70.30:+6.2f}  /  {100*fpr-36.39:+6.2f}  /  {roc_auc_score(y,p)-0.751:+.3f}')
print(f'published Marathi (their worst of 23): 82.43%')

rows = te.dataset.rows
pred = (p > 0.5).astype(int)

def by(title, groups):
    # A subset accuracy with no base rate is unreadable: `disputed` is 93.9%
    # `complete`, so 83.9% there is ten points BELOW answering `complete`
    # every time, while looking like the model's strongest bucket.
    print(f'\nby {title}:')
    for name, k in groups:
        if not k.any():
            continue
        base = max(y[k].sum(), k.sum() - y[k].sum()) / k.sum()
        a = (pred[k] == y[k]).mean()
        print(f'  {name:14s} n={k.sum():5d}  acc {100*a:6.2f}%   '
              f'baseline {100*base:6.2f}%   {100*(a-base):+6.2f}')

by('source', [(s_, np.array([r['source'] == s_ for r in rows]))
              for s_ in ('change', 'hold_intra', 'change_midseg', 'hold_inter')])
by('label agreement', [('agree', np.array([not r['dispute'] for r in rows])),
                       ('disputed', np.array([bool(r['dispute']) for r in rows]))])


## 5a. Capacity or data? — one forward pass

§5b below answers this with three training runs. This answers most of it with
one evaluation, which matters when Colab sessions are the scarce resource.

The model has already been selected on dev and scored on test. Now score it on
the data it *trained on*:

* **train ≈ dev** — it cannot even fit what it has. Capacity-limited. More data
  buys little; a larger encoder is the lever.
* **train ≫ dev** — it has memorised the training set. Overfitting. More data
  is the fix, and the 722 label-less recordings are back on the table.

Read the **gap**, not the absolute numbers.

In [ ]:
# `model` still holds best.pt, loaded in §5 above.
tr_ev = DataLoader(Clips('train'), batch_size=64, num_workers=2)
acc_tr, _, p_tr, y_tr = evaluate(tr_ev)
acc_dv, _, p_dv, y_dv = evaluate(dv)
gap = 100 * (acc_tr - acc_dv)

print(f'train {100*acc_tr:.2f}%   dev {100*acc_dv:.2f}%   gap {gap:+.2f} points')
print(f'  AUC  train {roc_auc_score(y_tr, p_tr):.4f}   dev {roc_auc_score(y_dv, p_dv):.4f}')

if gap < 5:
    print(f'\n  gap under 5 points -> CAPACITY-limited.')
    print('  More data is not the lever here; a larger encoder is. Do not spend')
    print('  15-20h diarizing the 722 recordings on this evidence.')
else:
    print(f'\n  gap over 5 points -> OVERFITTING.')
    print('  More data would help. The 722 label-less recordings are worth it.')
print('\n  (This is the cheap read. §5b measures the same thing properly,')
print('   at the cost of three more training runs.)')
ram('capacity check')

> **OPTIONAL — costs three training runs.** §5a above gives the cheap read.
> Skip both unless a session is free; Colab limits are the binding
> constraint, not GPU time.

## 5b. Learning curve — is data the bottleneck?

The question this answers: **would more data help, or is 8M parameters the
limit?** Train on 25% / 50% / 100% of the training set and read the slope.
Still climbing at 100% means collect more; flat means the encoder is saturated
and the next move is `whisper-base` instead.

Two things make this readable where the earlier `UNDISPUTED_ONLY` run was not:

**Scored on `hold_intra`, not on overall accuracy.** `change` and
`change_midseg` are 93% and 88% single-class — a speaker change all but implies
the turn ended — so 30% of the set is nearly free and dilutes any slope.
`hold_intra` is 58% of the data, near-balanced (majority baseline 50.3% on
test, 55.3% on dev), and is the actual EOT problem: same speaker,
mid-conversation, no structural cue. Overall 90% requires ~90% there, so it is
the only number with real information in it.

**Subsets are stratified by `(source, label)`,** so every fraction has the same
composition and the only thing varying is size. Dropping the disputed rows
changed the mixture as well as the count, which is why that result could not be
read as a data-size experiment.

Scored on **`dev`** — `test` stays sealed for the single run in §5. The slope is
what matters here, not the absolute.

Epochs are held at `EPOCHS` for every fraction, so the 25% run takes a quarter
of the optimiser steps. That is the honest "less data" condition; equalising
steps instead would just overfit the small subsets. Budget ~1.75× one full run.

**On reading the result:** dev `hold_intra` is 1,206 clips, so a single run's
accuracy carries roughly ±2.3 points of 95% sampling error. One gap between two
fractions is therefore weak evidence on its own — read the trend across all
three points, and if it looks marginal, re-run with `seed=1` in `stratified`
before concluding anything.


In [ ]:
from torch.utils.data import Subset
import random

FRACTIONS = (0.25, 0.50, 1.00)
CURVE_EPOCHS = EPOCHS          # same budget per fraction, see the note above

base_tr = Clips('train', UNDISPUTED_ONLY)
pos_of = {r['i']: k for k, r in enumerate(base_tr.rows)}
dv_rows = Clips('dev').rows
HOLD = np.array([r['source'] == 'hold_intra' for r in dv_rows])

def stratified(rows, frac, seed=0):
    """Same (source, label) mixture at every size, so only n varies."""
    g = {}
    for r in rows:
        g.setdefault((r['source'], r[LABEL]), []).append(r)
    rnd, out = random.Random(seed), []
    for k in sorted(g):
        v = sorted(g[k], key=lambda r: r['i'])
        rnd.shuffle(v)
        out += v[:max(1, round(frac * len(v)))]
    return out

@torch.no_grad()
def eval_model(m, loader):
    m.eval(); P, Y = [], []
    for x, y in loader:
        with torch.amp.autocast('cuda', enabled=(dev == 'cuda')):
            P.append(m(x.to(dev, non_blocking=True))['logits'].float().cpu())
        Y.append(y)
    return torch.cat(P).view(-1).numpy(), torch.cat(Y).numpy()

def run(frac):
    sub = stratified(base_tr.rows, frac)
    loader = DataLoader(Subset(base_tr, [pos_of[r['i']] for r in sub]),
                        batch_size=32, shuffle=True, num_workers=2,
                        pin_memory=True, drop_last=True)
    m = fresh_model()
    opt = AdamW(m.parameters(), lr=LR, weight_decay=0.01)
    steps = len(loader) * CURVE_EPOCHS
    sched = get_cosine_schedule_with_warmup(opt, int(0.2 * steps), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=(dev == 'cuda'))

    best, best_p = -1.0, None
    for ep in range(CURVE_EPOCHS):
        m.train()
        for k, (x, y) in enumerate(loader):
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(dev == 'cuda')):
                loss = m(x.to(dev, non_blocking=True), labels=y.to(dev))['loss']
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            if k % 50 == 0:
                print(f'\r  {frac:.0%} ep{ep} {k}/{len(loader)} loss {loss.item():.4f}', end='')
        p, y = eval_model(m, dv)
        # select on AUC, not accuracy: it is threshold-free, so the checkpoint
        # choice stops depending on where 0.5 happens to fall
        a = roc_auc_score(y, p)
        if a > best:
            best, best_p, best_y = a, p, y
    del m, opt; gc.collect(); torch.cuda.empty_cache()

    pred = (best_p > 0.5).astype(int)
    hi = (pred[HOLD] == best_y[HOLD]).mean()
    return {'frac': frac, 'n': len(sub), 'auc': best,
            'dev_all': (pred == best_y).mean(), 'dev_hold_intra': hi,
            'auc_hold_intra': roc_auc_score(best_y[HOLD], best_p[HOLD])}

curve = [run(f) for f in FRACTIONS]

_hy = np.array([r[LABEL] for r in dv_rows])[HOLD]
hb = max(_hy.sum(), len(_hy) - _hy.sum()) / len(_hy)
print(f'\n\ndev hold_intra: n={HOLD.sum()}, majority baseline {100*hb:.2f}%\n')
print(f'{"frac":>6} {"n train":>9} {"dev all":>9} {"hold_intra":>11} '
      f'{"vs base":>9} {"AUC(hi)":>9}')
for c in curve:
    print(f'{c["frac"]:>5.0%} {c["n"]:>9,} {100*c["dev_all"]:>8.2f}% '
          f'{100*c["dev_hold_intra"]:>10.2f}% {100*(c["dev_hold_intra"]-hb):>+8.2f} '
          f'{c["auc_hold_intra"]:>9.3f}')

d = 100 * (curve[-1]['dev_hold_intra'] - curve[-2]['dev_hold_intra'])
print(f'\nlast doubling of the data bought {d:+.2f} points on hold_intra.')
print(f'  (+-2.3 points of sampling error on n={HOLD.sum()}, so read all three)')
print('  clear rise across all three -> data-limited; collect more')
print('  flat from 50% to 100%      -> capacity-limited; try whisper-base first')
ram('after learning curve')


> **OPTIONAL — costs three training runs.** §5a above gives the cheap read.
> Skip both unless a session is free; Colab limits are the binding
> constraint, not GPU time.

## 5c. Learning rate — is 5e-5 right for batch 32?

Upstream trains at **batch 384, lr 5e-5**. This notebook uses **batch 32** and
kept the same learning rate, which is 12x fewer samples per step at the same
step size. Linear scaling puts the right value near `4e-6`, square-root scaling
near `1.4e-5` — so every run so far has been somewhere between 3.5x and 12x hot.

The dev curves show what that predicts: `77.25 -> 65.20 -> 75.74 -> 84.66` in
one run, a twelve-point collapse and full recovery. That is not a converging
optimiser, and it is the likely source of the ~2-point spread between runs of
identical config.

This matters for more than tidiness. The learning curve concluded *data-limited*
at a fixed 5e-5. The compute control holds regardless — more steps on 2,502
samples changed nothing — so the **slope** is real. But if the LR is wrong, all
four points were handicapped equally and the **ceiling** is unknown.

Every run below starts from the same seed, so init and shuffle are identical and
the only variable is the step size. Checkpoints are selected on dev **AUC**,
which is threshold-free. Three runs, ~25 min.


In [ ]:
from sklearn.metrics import roc_auc_score

LRS = (5e-5, 1.4e-5, 4e-6)     # as-is / sqrt-scaled / linear-scaled for batch 32
SWEEP_EPOCHS = 6           # SEED comes from §3 -- do not redefine it here

dv_rows = Clips('dev').rows
HOLD = np.array([r['source'] == 'hold_intra' for r in dv_rows])

@torch.no_grad()
def eval_dev(m):
    m.eval(); P, Y = [], []
    for x, y in dv:
        with torch.amp.autocast('cuda', enabled=(dev == 'cuda')):
            P.append(m(x.to(dev, non_blocking=True))['logits'].float().cpu())
        Y.append(y)
    return torch.cat(P).view(-1).numpy(), torch.cat(Y).numpy()

def sweep(lr):
    seed_everything(SEED)       # identical init AND shuffle order for every lr
    m = fresh_model()
    opt = AdamW(m.parameters(), lr=lr, weight_decay=0.01)
    steps = len(tr) * SWEEP_EPOCHS
    sched = get_cosine_schedule_with_warmup(opt, int(0.2 * steps), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=(dev == 'cuda'))

    best = (-1.0, None, None)
    for ep in range(SWEEP_EPOCHS):
        m.train()
        for k, (x, y) in enumerate(tr):
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(dev == 'cuda')):
                loss = m(x.to(dev, non_blocking=True), labels=y.to(dev))['loss']
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            if k % 100 == 0:
                print(f'\r  lr={lr:.1e} ep{ep} {k}/{len(tr)} loss {loss.item():.4f}', end='')
        p, y = eval_dev(m)
        a = roc_auc_score(y, p)
        # per-epoch dev is printed to expose instability, not just the endpoint
        print(f'\r  lr={lr:.1e} ep{ep}: dev {100*((p > 0.5).astype(int) == y).mean():6.2f}%'
              f'   AUC {a:.4f}          ')
        if a > best[0]:
            best = (a, p, y)
            torch.save(m.state_dict(), f'/content/lr_{lr:.0e}.pt')
    del m, opt; gc.collect(); torch.cuda.empty_cache()

    a, p, y = best
    pred = (p > 0.5).astype(int)
    return {'lr': lr, 'auc': a, 'dev': (pred == y).mean(),
            'hi': (pred[HOLD] == y[HOLD]).mean(),
            'auc_hi': roc_auc_score(y[HOLD], p[HOLD])}

res = [sweep(lr) for lr in LRS]

print(f'\n{"lr":>9} {"dev all":>9} {"hold_intra":>11} {"AUC":>7} {"AUC(hi)":>9}')
for r in res:
    print(f'{r["lr"]:>9.1e} {100*r["dev"]:>8.2f}% {100*r["hi"]:>10.2f}% '
          f'{r["auc"]:>7.4f} {r["auc_hi"]:>9.4f}')
b = max(res, key=lambda r: r['auc_hi'])
print(f'\nbest by dev AUC(hold_intra): lr={b["lr"]:.1e} -> /content/lr_{b["lr"]:.0e}.pt')
print(f'  vs 5e-5: {100*(b["hi"] - res[0]["hi"]):+.2f} points hold_intra, '
      f'{b["auc_hi"] - res[0]["auc_hi"]:+.4f} AUC')
print('  (+-2.3 points of sampling error on n=1206 -- trust AUC over accuracy here)')

# /content is ephemeral. The 2026-08-22 run picked 5e-5 here and then lost the
# checkpoint with the session, because only the ONNX was copied out -- so the
# shipped model stayed at 1.4e-5, the LR this sweep had just rejected.
!cp /content/lr_*.pt {SRC}/ 2>/dev/null; ls -la {SRC}/lr_*.pt

# Promote the winner so §5/§6 below export the model this sweep chose, rather
# than whatever §5 happened to leave in `best.pt`.
PROMOTE = True
if PROMOTE:
    import shutil
    shutil.copy(f'/content/lr_{b["lr"]:.0e}.pt', '/content/best.pt')
    print(f'\n  promoted lr={b["lr"]:.1e} to /content/best.pt'
          '  -- RE-RUN §5 (test) and §6 (export) now')
ram('after lr sweep')


## 6. Export

ONNX so it drops into Pipecat / LiveKit exactly like the released checkpoints.
The graph ends in a Sigmoid and the output is named `logits`, matching
upstream's convention — so anything that runs `smart-turn-v3.2-cpu.onnx` runs
this unchanged. Expect ~32 MB, the same as the released fp32 checkpoint.

In [ ]:
# the legacy exporter serialises via the `onnx` package, which Colab
# does not ship by default
!pip -q install onnx

class Exportable(nn.Module):
    def __init__(self, m):
        super().__init__(); self.inner = m
    def forward(self, input_features):
        return self.inner(input_features)['logits']

# imported here rather than inherited from the zero-shot cell: the export must
# still verify itself if that section is skipped or deleted
import onnxruntime as ort

model.eval()
ex = Exportable(model).cpu().eval()
# dynamo=False: the torch>=2.9 default exporter pulls in onnxscript, and the
# legacy path produces a graph that matches the released fp32 checkpoint.
torch.onnx.export(ex, torch.randn(1, 80, 800), '/content/smart-turn-tamil.onnx',
                  dynamo=False, input_names=['input_features'], output_names=['logits'],
                  opset_version=18,
                  dynamic_axes={'input_features': {0: 'batch'}, 'logits': {0: 'batch'}})

# The export has to agree with the torch model, or the shipped artefact is not
# the thing that was measured above.
s2 = ort.InferenceSession('/content/smart-turn-tamil.onnx', providers=['CPUExecutionProvider'])
Xt = np.ascontiguousarray(FEATS['test'][:64], dtype=np.float32)
po = s2.run(None, {'input_features': Xt})[0].reshape(-1)
with torch.no_grad():
    pt = ex(torch.from_numpy(Xt)).view(-1).numpy()
print(f'onnx vs torch max abs diff: {np.abs(po - pt).max():.2e}')
assert np.abs(po - pt).max() < 1e-4, 'ONNX export does not match the torch model'

model.to(dev)
!ls -la /content/smart-turn-tamil.onnx
!cp /content/smart-turn-tamil.onnx /content/best.pt {SRC}/
print('copied back to Drive')